# Project 2 Milestone 5: DBSCAN Robustness and Parameter Sensitivity

This notebook evaluates whether the candidate-enriched DBSCAN outlier population found in Project 2 Milestone 4 is stable across different DBSCAN parameter settings.

The goal is not to claim a confirmed astrophysical substructure, but to test whether the candidate-rich outlier behavior is robust in feature space.

## Research Question

Is the candidate-enriched outlier population stable across different DBSCAN parameter settings in PCA and UMAP embedding spaces?

## Step 1: Load PCA and UMAP Embeddings

We first load the two embedding tables produced in previous milestones. PCA provides a linear baseline view of the combined chemo-kinematic feature space, while UMAP provides a nonlinear local-neighborhood view.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

PCA_PATH = Path('../data/processed/project2_combined_chemo_kinematic_pca_embedding.csv')
UMAP_PATH = Path('../data/processed/project2_combined_chemo_kinematic_umap_embedding.csv')

pca_df = pd.read_csv(PCA_PATH)
umap_df = pd.read_csv(UMAP_PATH)

print('PCA shape:', pca_df.shape)
print('UMAP shape:', umap_df.shape)

print('\nPCA columns:')
print(list(pca_df.columns))

print('\nUMAP columns:')
print(list(umap_df.columns))

pca_df.head()


## Step 2: Define DBSCAN Helper Functions

We define reusable helper functions for running DBSCAN and summarizing the noise/outlier population. This allows the same logic to be applied consistently across many parameter settings.

In [ ]:
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN

def run_dbscan(dataframe, coordinate_cols, eps, min_samples):
    X = dataframe[coordinate_cols].to_numpy()
    X_scaled = StandardScaler().fit_transform(X)
    model = DBSCAN(eps=eps, min_samples=min_samples)
    return model.fit_predict(X_scaled)


def summarize_noise_population(dataframe, labels, embedding_name, eps, min_samples):
    labelled = dataframe.copy()
    labelled['dbscan_label'] = labels

    noise = labelled[labelled['dbscan_label'] == -1]
    n_total = len(labelled)
    n_noise = len(noise)
    n_clusters = len(set(labels)) - (1 if -1 in set(labels) else 0)

    if n_noise == 0:
        return {
            'embedding': embedding_name,
            'eps': eps,
            'min_samples': min_samples,
            'n_clusters': n_clusters,
            'n_noise': 0,
            'noise_fraction': 0.0,
            'noise_candidate_count': 0,
            'noise_candidate_fraction': np.nan,
            'noise_mean_feh': np.nan,
            'noise_mean_galcen_vtot_kms': np.nan,
        }

    return {
        'embedding': embedding_name,
        'eps': eps,
        'min_samples': min_samples,
        'n_clusters': n_clusters,
        'n_noise': n_noise,
        'noise_fraction': n_noise / n_total,
        'noise_candidate_count': int(noise['chemo_kinematic_candidate'].sum()),
        'noise_candidate_fraction': noise['chemo_kinematic_candidate'].mean(),
        'noise_mean_feh': noise['feh'].mean(),
        'noise_mean_galcen_vtot_kms': noise['galcen_vtot_kms'].mean(),
    }


## Step 3: Small PCA Parameter Sweep

Before running a broad robustness scan, we test a small PCA-only DBSCAN parameter grid. This helps show how `eps` and `min_samples` affect the size and candidate enrichment of the noise population.

In [ ]:
small_eps_values = [0.20, 0.25, 0.30]
small_min_samples_values = [5, 8, 12]

small_rows = []

for eps in small_eps_values:
    for min_samples in small_min_samples_values:
        labels = run_dbscan(
            pca_df,
            ['combined_chemo_kinematic_pc1', 'combined_chemo_kinematic_pc2'],
            eps=eps,
            min_samples=min_samples,
        )
        small_rows.append(
            summarize_noise_population(
                pca_df,
                labels,
                embedding_name='pca',
                eps=eps,
                min_samples=min_samples,
            )
        )

small_pca_sweep = pd.DataFrame(small_rows)
small_pca_sweep.sort_values(['eps', 'min_samples'])


## Step 4: Small UMAP Parameter Sweep

We repeat the same small DBSCAN parameter sweep on the UMAP embedding. This allows a direct comparison between PCA and UMAP behavior under the same parameter choices.

In [ ]:
small_umap_rows = []

for eps in small_eps_values:
    for min_samples in small_min_samples_values:
        labels = run_dbscan(
            umap_df,
            ['umap_1', 'umap_2'],
            eps=eps,
            min_samples=min_samples,
        )
        small_umap_rows.append(
            summarize_noise_population(
                umap_df,
                labels,
                embedding_name='umap',
                eps=eps,
                min_samples=min_samples,
            )
        )

small_umap_sweep = pd.DataFrame(small_umap_rows)
small_umap_sweep.sort_values(['eps', 'min_samples'])


## Step 5: Save Small Parameter-Sweep Summary

We combine the PCA and UMAP small parameter-sweep results into a single summary table. This preserves the first robustness comparison as a reusable project output.

In [ ]:
OUTPUT_DIR = Path('../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

small_parameter_sweep_summary = pd.concat(
    [small_pca_sweep, small_umap_sweep],
    ignore_index=True,
)

SMALL_SWEEP_PATH = OUTPUT_DIR / 'project2_dbscan_small_parameter_sweep_summary.csv'
small_parameter_sweep_summary.to_csv(SMALL_SWEEP_PATH, index=False)

print('Saved:', SMALL_SWEEP_PATH)
small_parameter_sweep_summary.sort_values(['embedding', 'eps', 'min_samples'])


## Step 6: Interpretation of the Small Parameter Sweep

The small PCA parameter sweep shows a stable candidate-rich noise/outlier population. Across the tested PCA DBSCAN settings, 24–25 of the 27 Project 1 candidates are repeatedly assigned to the noise population.

This behavior is not reproduced in the small UMAP parameter sweep. Under the same parameter choices, UMAP DBSCAN assigns almost all stars to a single large cluster, leaving only 0–2 stars as noise and 0–1 Project 1 candidates in the noise population.

This contrast suggests that the candidate-rich outlier behavior is more visible in the PCA-projected global feature space than in the current UMAP local-neighborhood embedding. The result should be interpreted as feature-space evidence, not as confirmation of a distinct astrophysical population.